# NBGrader - Sistema Automatizado de Calificación

## Sistema Dinámico y Configurable

Este notebook permite configurar cualquier curso, assignment y lista de estudiantes de forma dinámica.

## Observaciones Importantes:
1. Se debe reiniciar la sesión si se copian archivos nuevos de estudiantes
2. Los nombres de carpetas NO pueden tener acentos (el feedback no funciona)
3. Todos los parámetros son configurables al inicio

## Workflow:
1. Configurar parámetros del curso
2. Cargar estudiantes desde CSV
3. Generar assignment
4. Calificar automáticamente
5. Generar feedback y exportar notas

## 1. Instalación de Dependencias

In [ ]:
# Instalar nbclient 0.6.1
!pip install nbclient==0.6.1 -q

In [ ]:
# Instalar nbgrader 0.8.1
!pip install nbgrader==0.8.1 -q

## 2. Montar Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## 3. Configuración Dinámica del Curso

**IMPORTANTE:** Configura aquí todos los parámetros de tu curso

In [ ]:
import os
import pandas as pd
import shutil
import re
from pathlib import Path

# ============================================================================
# CONFIGURACIÓN DINÁMICA - PERSONALIZA ESTOS VALORES
# ============================================================================

# 1. RUTA BASE EN GOOGLE DRIVE
# Esta es la carpeta principal donde se organizarán TODOS tus cursos
# Ejemplos: '/content/drive/MyDrive/nbgrader', '/content/drive/MyDrive/Cursos'
print("💡 La ruta base es la carpeta principal para TODOS tus cursos")
print("   Dentro se creará una subcarpeta para cada curso")
print("   Ejemplo: /content/drive/MyDrive/nbgrader\n")
BASE_PATH_INPUT = input("Ingresa la ruta base en Google Drive: ").strip()
if not BASE_PATH_INPUT:
    BASE_PATH_INPUT = '/content/drive/MyDrive/nbgrader'
    print(f"⚠️  Usando valor por defecto: {BASE_PATH_INPUT}")

# 2. NOMBRE DEL CURSO (se crea como carpeta dentro de BASE_PATH)
# Ejemplos: 'Python_AP', 'DataScience_101', 'MachineLearning_2024'
print("\n💡 El nombre del curso se creará como carpeta dentro de la ruta base")
print("   Ejemplo: Si usas 'Python_2024', se creará:")
print(f"   {BASE_PATH_INPUT}/Python_2024/\n")
COURSE_ID = input("Ingresa el ID del curso (ej: Python_AP): ").strip()
if not COURSE_ID:
    COURSE_ID = 'MiCurso'
    print(f"⚠️  Usando valor por defecto: {COURSE_ID}")

# 3. ID DEL ASSIGNMENT
# Ejemplos: 'S01_D02_A02', 'Tarea1', 'Parcial_Final'
ASSIGNMENT_ID = input("\nIngresa el ID del assignment (ej: S01_D02_A02): ").strip()
if not ASSIGNMENT_ID:
    ASSIGNMENT_ID = 'Assignment1'
    print(f"⚠️  Usando valor por defecto: {ASSIGNMENT_ID}")

# 4. TIMEOUT DE EJECUCIÓN (en segundos)
# Recomendado: 180 (3 minutos) - Aumentar si los ejercicios son complejos
TIMEOUT = input("\nTimeout de ejecución en segundos [180]: ").strip()
TIMEOUT = int(TIMEOUT) if TIMEOUT else 180

# Validar que no haya espacios ni caracteres especiales
if ' ' in COURSE_ID or ' ' in ASSIGNMENT_ID:
    print("\n❌ ERROR: Los IDs no pueden contener espacios")
    print("   Usa guiones bajos en su lugar: Mi_Curso en vez de 'Mi Curso'")
    raise ValueError("IDs con espacios no permitidos")

# CONSTRUIR RUTA COMPLETA: BASE_PATH/COURSE_ID
BASE_PATH = os.path.join(BASE_PATH_INPUT, COURSE_ID)

# Rutas de directorios (dentro de BASE_PATH/COURSE_ID/)
DIRS = {
    'source': os.path.join(BASE_PATH, 'source'),
    'release': os.path.join(BASE_PATH, 'release'),
    'submitted': os.path.join(BASE_PATH, 'submitted'),
    'autograded': os.path.join(BASE_PATH, 'autograded'),
    'feedback': os.path.join(BASE_PATH, 'feedback')
}

print("\n" + "="*70)
print("CONFIGURACIÓN DEL CURSO")
print("="*70)
print(f"📚 Curso:       {COURSE_ID}")
print(f"📝 Assignment:  {ASSIGNMENT_ID}")
print(f"📂 Ruta base:   {BASE_PATH_INPUT}")
print(f"📁 Ruta curso:  {BASE_PATH}")
print(f"⏱️  Timeout:     {TIMEOUT} segundos")
print("="*70)
print(f"\n💡 Estructura que se creará:")
print(f"   {BASE_PATH}/")
print(f"   ├── source/")
print(f"   ├── release/")
print(f"   ├── submitted/")
print(f"   ├── autograded/")
print(f"   └── feedback/")
print("="*70)
print("\n✅ Configuración lista!")

## 4. Generar Archivo de Configuración de NBGrader

Esta celda crea automáticamente el archivo `nbgrader_config.py` con tu configuración.

In [ ]:
# Generar nbgrader_config.py dinámicamente
config_content = f'''# ============================================================================
# NBGrader Configuration File - Generado Automáticamente
# ============================================================================
# Curso: {COURSE_ID}
# Assignment: {ASSIGNMENT_ID}
# ============================================================================

# Directorio raíz del curso
c.CourseDirectory.root = '{BASE_PATH}'

# Configuración de ejecución
c.Execute.timeout = {TIMEOUT}
c.Execute.ipython_hist_file = ':memory:'
c.Execute.record_timing = True

# Configuración de soluciones y tests
c.ClearSolutions.begin_solution_delimeter = 'BEGIN SOLUTION'
c.ClearSolutions.end_solution_delimeter = 'END SOLUTION'
c.ClearSolutions.code_stub = {{
    'python': '# YOUR CODE HERE\\nraise NotImplementedError()',
    'r': '# YOUR CODE HERE\\nstop("No Answer Given!")'
}}

# Tests ocultos
c.ClearHiddenTests.begin_test_delimeter = 'BEGIN HIDDEN TESTS'
c.ClearHiddenTests.end_test_delimeter = 'END HIDDEN TESTS'
c.ClearHiddenTests.enforce_metadata = False

# Bloqueo de celdas
c.LockCells.lock_grade_cells = True
c.LockCells.lock_readonly_cells = True
c.LockCells.lock_solution_cells = True

# Límites de output
c.LimitOutput.max_lines = 1000
c.LimitOutput.max_traceback = 100

# Archivos a ignorar
c.CourseDirectory.ignore = [
    '.ipynb_checkpoints',
    '*.pyc',
    '__pycache__',
    'feedback',
    '.DS_Store'
]

# Tamaño máximo de archivos (100 MB)
c.CourseDirectory.max_file_size = 100000

# Penalizaciones por entrega tardía
c.LateSubmissionPlugin.penalty_method = 'none'

# Timestamps
c.Exchange.timestamp_format = '%Y-%m-%d %H:%M:%S %Z'
c.Exchange.timezone = 'UTC'

# Feedback
c.GetGrades.display_data_priority = [
    'text/html',
    'application/pdf',
    'text/latex',
    'image/svg+xml',
    'image/png',
    'image/jpeg',
    'text/plain'
]
'''

# Escribir archivo en /content/ (ubicación principal)
config_path_content = '/content/nbgrader_config.py'
with open(config_path_content, 'w') as f:
    f.write(config_content)

# IMPORTANTE: También copiar al directorio del curso para que nbgrader lo encuentre
config_path_course = os.path.join(BASE_PATH, 'nbgrader_config.py')
with open(config_path_course, 'w') as f:
    f.write(config_content)

print(f"✅ Archivo de configuración generado en:")
print(f"   1. {config_path_content}")
print(f"   2. {config_path_course}")
print(f"\n📋 Contenido:")
print("="*70)
!head -20 /content/nbgrader_config.py
print("...")
print("="*70)

In [ ]:
# DIAGNÓSTICO: Verificar ubicación del archivo de configuración
import os

print("="*70)
print("🔍 DIAGNÓSTICO DE nbgrader_config.py")
print("="*70)

print("\n1️⃣ Directorio de trabajo actual:")
print(f"   {os.getcwd()}")

print("\n2️⃣ Ubicaciones donde debería estar el archivo:")
config_locations = [
    ('/content/nbgrader_config.py', 'Ubicación estándar en Colab'),
    (os.path.join(BASE_PATH, 'nbgrader_config.py'), 'Ubicación en el directorio del curso'),
]

for path, desc in config_locations:
    exists = "✅ EXISTE" if os.path.exists(path) else "❌ NO EXISTE"
    print(f"\n   {exists}")
    print(f"   Ruta: {path}")
    print(f"   Desc: {desc}")

print("\n3️⃣ Ejecutar nbgrader con --debug para ver dónde busca:")
print("-"*70)

# Cambiar al directorio del curso
os.chdir(BASE_PATH)
print(f"Cambiado a directorio: {os.getcwd()}")

# Ejecutar con --debug para ver dónde busca
print("\nEjecutando: nbgrader db assignment list --debug")
print("-"*70)
!nbgrader db assignment list --debug 2>&1 | grep -i "config" || echo "No se encontró información sobre config en el output"

# Volver a /content
os.chdir('/content')

print("\n" + "="*70)
print("💡 SOLUCIÓN:")
print("="*70)
print("Si ves 'No nbgrader_config.py file found', intenta:")
print("1. Re-ejecutar la celda 4 (Generar Archivo de Configuración)")
print("2. Verificar que BASE_PATH está correctamente configurado")
print("3. El warning es INFORMATIVO - si el archivo existe en BASE_PATH,")
print("   los comandos deberían funcionar correctamente")
print("="*70)

## 5. Crear Estructura de Directorios

In [ ]:
# Crear directorios base del curso
print("Creando estructura de directorios...\n")
for dir_name, dir_path in DIRS.items():
    try:
        os.makedirs(dir_path, exist_ok=True)
        print(f"✓ {dir_name:12} → {dir_path}")
    except OSError as error:
        print(f"✗ {dir_name:12} → Error: {error}")

print(f"\n✅ Estructura creada en: {BASE_PATH}")

## 6. Cargar Lista de Estudiantes desde CSV

### Formato del CSV esperado:
```csv
nombre_estudiante
Apellido1_Apellido2_Nombre1_Nombre2
...
```

**IMPORTANTE:** Los nombres NO deben tener acentos ni caracteres especiales.

In [ ]:
from google.colab import files

# Subir archivo CSV
print("📤 Por favor, sube el archivo CSV con la lista de estudiantes")
print("   Formato: nombre_estudiante (una columna, sin acentos)\n")
uploaded = files.upload()

# Leer CSV
csv_filename = list(uploaded.keys())[0]
df_estudiantes = pd.read_csv(csv_filename)

# Validar que existe la columna requerida
if 'nombre_estudiante' not in df_estudiantes.columns:
    raise ValueError("❌ El CSV debe tener una columna llamada 'nombre_estudiante'")

# Obtener lista de estudiantes
estudiantes = df_estudiantes['nombre_estudiante'].tolist()

# Validar nombres (sin acentos ni espacios)
print("\n🔍 Validando nombres de estudiantes...\n")
errores_validacion = []

for est in estudiantes:
    # Convertir a string por si acaso
    est_str = str(est).strip()
    
    # Verificar espacios
    if ' ' in est_str:
        errores_validacion.append(f"   ⚠️  '{est_str}' contiene espacios (usar guiones bajos)")
    
    # Verificar caracteres especiales o acentos
    if not re.match(r'^[a-zA-Z0-9_]+$', est_str):
        errores_validacion.append(f"   ⚠️  '{est_str}' contiene acentos o caracteres especiales")

if errores_validacion:
    print("❌ ERRORES DE VALIDACIÓN:")
    for error in errores_validacion:
        print(error)
    print("\n⚠️  Los nombres deben:")
    print("   - Usar guiones bajos en lugar de espacios")
    print("   - NO tener acentos (ej: Martinez en vez de Martínez)")
    print("   - Solo letras, números y guiones bajos")
    raise ValueError("CSV con nombres inválidos")

print("="*70)
print(f"✅ Se cargaron {len(estudiantes)} estudiantes válidos")
print("="*70)
for i, est in enumerate(estudiantes, 1):
    print(f"  {i:2}. {est}")
print("="*70)

### Opción alternativa: Definir estudiantes manualmente

In [ ]:
# Descomentar si prefieres definir la lista manualmente
# estudiantes = [
#     'Estudiante_Uno',
#     'Estudiante_Dos',
#     'Estudiante_Tres'
# ]
# print(f"✓ Lista manual: {len(estudiantes)} estudiantes")

## 7. Crear Carpetas de Estudiantes

In [ ]:
# Crear carpetas para cada estudiante en 'submitted'
submitted_base = DIRS['submitted']

print(f"Creando carpetas para {len(estudiantes)} estudiantes...\n")

for estudiante in estudiantes:
    # Crear carpeta del estudiante
    student_path = os.path.join(submitted_base, estudiante)
    os.makedirs(student_path, exist_ok=True)
    
    # Crear subcarpeta del assignment
    assignment_path = os.path.join(student_path, ASSIGNMENT_ID)
    os.makedirs(assignment_path, exist_ok=True)
    
    print(f"✓ {estudiante}/{ASSIGNMENT_ID}")

print(f"\n✅ Carpetas creadas en: {submitted_base}")

## 8. Generar Assignment para Estudiantes

**Prerequisito:** Debes tener el notebook fuente en:
```
source/{ASSIGNMENT_ID}/{ASSIGNMENT_ID}.ipynb
```

### ⚠️ IMPORTANTE: Configurar Test Cells para que NO aparezcan en el Feedback

**Problema:** Los estudiantes ven el código de los tests en el feedback HTML.

**Solución:** Configura CORRECTAMENTE los metadatos de las celdas de test en tu notebook fuente ANTES de generar el assignment.

#### ✅ Configuración CORRECTA para ocultar tests:

En el notebook fuente (`source/{ASSIGNMENT_ID}/{ASSIGNMENT_ID}.ipynb`), las celdas de test deben tener estos metadatos:

```json
{
  "nbgrader": {
    "grade": true,
    "grade_id": "test_cell_1",
    "locked": true,
    "points": 10,
    "schema_version": 3,
    "solution": false,
    "task": false
  }
}
```

**Campos críticos:**
- ✅ `"grade": true` - Marca como celda de calificación
- ✅ `"locked": true` - **CRITICAL:** Oculta el código del test del feedback
- ✅ `"points": 10` - Puntos que vale el test (cambiar según necesites)
- ✅ `"solution": false` - No es una celda de solución

#### 🔧 Cómo editar metadatos en Jupyter/Colab:

**En Jupyter Notebook:**
1. Selecciona la celda de test
2. Click en: `View` → `Cell Toolbar` → `Edit Metadata`
3. Click en `Edit Metadata` en la celda
4. Pega la configuración JSON de arriba
5. Click `Edit` para guardar

**En Google Colab:**
1. Abre el notebook en Jupyter local o usa el editor nbgrader
2. Los metadatos no son fácilmente editables en Colab directo
3. **Recomendación:** Usa Jupyter local para crear el assignment

#### 📋 Ejemplo de celda de test completa:

**Código de la celda:**
```python
# Test Cell - Los estudiantes NO verán este código en el feedback
assert suma(2, 3) == 5, "La suma de 2+3 debe ser 5"
assert suma(0, 0) == 0, "La suma de 0+0 debe ser 0"
assert suma(-1, 1) == 0, "La suma de -1+1 debe ser 0"

# BEGIN HIDDEN TESTS
# Estos tests adicionales TAMPOCO se mostrarán
assert suma(100, 200) == 300
assert suma(-50, -50) == -100
# END HIDDEN TESTS
```

**Metadatos de la celda:**
```json
{
  "nbgrader": {
    "grade": true,
    "grade_id": "test_suma",
    "locked": true,
    "points": 2,
    "solution": false
  }
}
```

**Resultado en el feedback del estudiante:**
- ✅ Los estudiantes verán: **"Test test_suma: PASSED (2/2 points)"** o **"FAILED (0/2 points)"**
- ❌ Los estudiantes NO verán: El código `assert suma(2, 3) == 5...`

#### ⚠️ Qué pasa si NO configuras `locked: true`:

- ❌ Los estudiantes verán TODO el código del test
- ❌ Podrán copiar la solución directamente
- ❌ No tiene sentido evaluar si ven las respuestas

#### 🔍 Verificar que está funcionando:

Después de generar el assignment (celda 19), verifica el notebook en `release/`:

```python
# Ejecuta esto después de generar el assignment
import json
import nbformat

release_nb_path = os.path.join(DIRS['release'], ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")
nb = nbformat.read(release_nb_path, as_version=4)

print("Celdas de test en el release:")
for i, cell in enumerate(nb.cells):
    if cell.get('metadata', {}).get('nbgrader', {}).get('grade', False):
        grade_id = cell['metadata']['nbgrader'].get('grade_id', 'unknown')
        locked = cell['metadata']['nbgrader'].get('locked', False)
        points = cell['metadata']['nbgrader'].get('points', 0)
        has_code = len(cell.get('source', '')) > 0
        
        print(f"  Celda {i}: {grade_id}")
        print(f"    - Locked: {locked} {'✅' if locked else '❌ PROBLEMA!'}")
        print(f"    - Points: {points}")
        print(f"    - Tiene código: {has_code}")
        if not locked and has_code:
            print(f"    ⚠️  WARNING: Esta celda NO está bloqueada. Los estudiantes verán el código!")
```

In [ ]:
# VERIFICAR CONFIGURACIÓN DE TEST CELLS EN EL NOTEBOOK FUENTE
# Ejecuta esta celda ANTES de generar el assignment para verificar
# que las celdas de test están correctamente configuradas

import json
import nbformat
import os

source_nb_path = os.path.join(DIRS['source'], ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")

if not os.path.exists(source_nb_path):
    print(f"❌ No se encuentra el notebook fuente: {source_nb_path}")
    print(f"\n💡 Primero debes crear el notebook fuente en:")
    print(f"   {os.path.dirname(source_nb_path)}/")
else:
    print("="*70)
    print("🔍 VERIFICACIÓN DE CELDAS DE TEST")
    print("="*70)
    print(f"\nNotebook: {source_nb_path}\n")
    
    nb = nbformat.read(source_nb_path, as_version=4)
    
    test_cells = []
    solution_cells = []
    problemas = []
    
    for i, cell in enumerate(nb.cells):
        metadata = cell.get('metadata', {})
        nbgrader_meta = metadata.get('nbgrader', {})
        
        # Identificar celdas de test (grade = true)
        if nbgrader_meta.get('grade', False) and not nbgrader_meta.get('solution', False):
            grade_id = nbgrader_meta.get('grade_id', 'unknown')
            locked = nbgrader_meta.get('locked', False)
            points = nbgrader_meta.get('points', 0)
            has_code = cell.cell_type == 'code' and len(cell.get('source', '').strip()) > 0
            
            test_cells.append({
                'index': i,
                'grade_id': grade_id,
                'locked': locked,
                'points': points,
                'has_code': has_code
            })
            
            # Verificar si hay problemas
            if has_code and not locked:
                problemas.append({
                    'index': i,
                    'grade_id': grade_id,
                    'problema': 'NO LOCKED - Los estudiantes verán el código del test'
                })
        
        # Identificar celdas de solución
        if nbgrader_meta.get('solution', False):
            solution_cells.append({
                'index': i,
                'grade_id': nbgrader_meta.get('grade_id', 'unknown')
            })
    
    # Mostrar resultados
    print("📊 RESUMEN:")
    print(f"   Celdas de test encontradas: {len(test_cells)}")
    print(f"   Celdas de solución encontradas: {len(solution_cells)}")
    print(f"   Problemas detectados: {len(problemas)}")
    
    if test_cells:
        print("\n" + "="*70)
        print("📝 CELDAS DE TEST:")
        print("="*70)
        for tc in test_cells:
            status = "✅" if tc['locked'] else "❌"
            print(f"\n  {status} Celda {tc['index']}: {tc['grade_id']}")
            print(f"     - Locked: {tc['locked']}")
            print(f"     - Points: {tc['points']}")
            print(f"     - Tiene código: {tc['has_code']}")
            
            if tc['has_code'] and not tc['locked']:
                print(f"     ⚠️  PROBLEMA: Esta celda NO está bloqueada!")
                print(f"     ⚠️  Los estudiantes VERÁN el código en el feedback!")
    
    if problemas:
        print("\n" + "="*70)
        print("⚠️  PROBLEMAS DETECTADOS:")
        print("="*70)
        for p in problemas:
            print(f"\n  ❌ Celda {p['index']}: {p['grade_id']}")
            print(f"     {p['problema']}")
        
        print("\n" + "="*70)
        print("🔧 SOLUCIÓN:")
        print("="*70)
        print("1. Abre el notebook fuente en Jupyter")
        print("2. Para cada celda de test marcada con ❌:")
        print("   a. Selecciona la celda")
        print("   b. View → Cell Toolbar → Edit Metadata")
        print("   c. En los metadatos, asegúrate de tener:")
        print('      "locked": true,')
        print("   d. Guarda los cambios")
        print("3. Re-ejecuta esta celda de verificación")
        print("4. Cuando todas las celdas tengan ✅, ejecuta la celda de generar assignment")
    else:
        print("\n" + "="*70)
        print("✅ CONFIGURACIÓN CORRECTA")
        print("="*70)
        print("Todas las celdas de test están correctamente configuradas.")
        print("Los estudiantes NO verán el código de los tests en el feedback.")
        print("\n💡 Puedes continuar con la siguiente celda para generar el assignment.")
    
    print("\n" + "="*70)

In [ ]:
# Verificar que existe el archivo fuente
source_nb_path = os.path.join(DIRS['source'], ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")

if not os.path.exists(source_nb_path):
    print(f"❌ ERROR: No se encuentra el notebook fuente")
    print(f"\n📁 Esperado en: {source_nb_path}")
    print(f"\n💡 Debes crear primero el assignment maestro y subirlo a:")
    print(f"   {os.path.dirname(source_nb_path)}/")
else:
    print(f"✅ Archivo fuente encontrado: {source_nb_path}")
    print(f"\n🚀 Generando assignment para estudiantes...\n")
    print("="*70)
    
    # IMPORTANTE: Cambiar al directorio del curso para que nbgrader encuentre el config
    import os
    os.chdir(BASE_PATH)
    
    # Generar assignment - usar assignment como argumento posicional
    !nbgrader generate_assignment '{ASSIGNMENT_ID}' --debug
    
    # Volver a /content
    os.chdir('/content')
    
    print("="*70)
    release_path = os.path.join(DIRS['release'], ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")
    print(f"\n✅ Assignment generado en: {release_path}")
    print(f"\n💡 Comparte este archivo con tus estudiantes")

## 9. Autograding Masivo

**Prerequisito:** Los estudiantes deben haber subido sus notebooks en:
```
submitted/{estudiante}/{ASSIGNMENT_ID}/{ASSIGNMENT_ID}.ipynb
```

In [ ]:
# Verificar que hay submissions
print("🔍 Verificando submissions...\n")

submissions_encontradas = []
submissions_faltantes = []

for estudiante in estudiantes:
    nb_path = os.path.join(DIRS['submitted'], estudiante, ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")
    if os.path.exists(nb_path):
        submissions_encontradas.append(estudiante)
        print(f"✓ {estudiante}")
    else:
        submissions_faltantes.append(estudiante)
        print(f"✗ {estudiante} - NO ENCONTRADO")

print(f"\n📊 Resumen:")
print(f"   ✓ Submissions encontradas: {len(submissions_encontradas)}/{len(estudiantes)}")
print(f"   ✗ Submissions faltantes:   {len(submissions_faltantes)}/{len(estudiantes)}")

if submissions_faltantes:
    print(f"\n⚠️  Estudiantes sin submission:")
    for est in submissions_faltantes:
        print(f"   - {est}")

if not submissions_encontradas:
    print(f"\n❌ No hay submissions para calificar")
else:
    print(f"\n✅ Listo para calificar {len(submissions_encontradas)} estudiantes")

In [ ]:
# Autograding para todos los estudiantes con submissions
import subprocess
import json
import os

if not submissions_encontradas:
    print("❌ No hay submissions para calificar")
else:
    errores = []
    exitosos = []

    print(f"\n🚀 Iniciando calificación de {len(submissions_encontradas)} estudiantes...\n")
    print("="*70)

    # IMPORTANTE: Cambiar al directorio del curso para que nbgrader encuentre el config
    os.chdir(BASE_PATH)

    for i, estudiante in enumerate(submissions_encontradas, 1):
        print(f"\n[{i}/{len(submissions_encontradas)}] 📝 Calificando: {estudiante}")
        print("-"*70)
        
        try:
            # IMPORTANTE: Usar el formato correcto para nbgrader 0.8.1
            # nbgrader autograde <assignment> --student <student_id> --force
            cmd = f"nbgrader autograde '{ASSIGNMENT_ID}' --student '{estudiante}' --force"
            result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
            
            if result.returncode == 0:
                exitosos.append(estudiante)
                print(f"✅ {estudiante} - OK")
                
                # Verificar que el puntaje se guardó en la base de datos
                check_cmd = f"nbgrader db assignment list --student '{estudiante}'"
                check_result = subprocess.run(check_cmd, shell=True, capture_output=True, text=True)
                if check_result.returncode == 0 and ASSIGNMENT_ID in check_result.stdout:
                    print(f"   ✓ Registrado en base de datos")
            else:
                errores.append(estudiante)
                print(f"❌ {estudiante} - ERROR (código: {result.returncode})")
                if result.stderr:
                    print(f"   Stderr: {result.stderr[:300]}")
                if result.stdout:
                    print(f"   Stdout: {result.stdout[:300]}")
        except Exception as e:
            errores.append(estudiante)
            print(f"❌ {estudiante} - EXCEPTION: {e}")

    # Volver a /content
    os.chdir('/content')

    print("\n" + "="*70)
    print("📊 RESUMEN DE CALIFICACIÓN")
    print("="*70)
    print(f"✅ Exitosos: {len(exitosos)}/{len(submissions_encontradas)}")
    print(f"❌ Errores:  {len(errores)}/{len(submissions_encontradas)}")

    if exitosos:
        print(f"\n✅ Estudiantes calificados exitosamente:")
        for est in exitosos:
            print(f"   ✓ {est}")
    
    if errores:
        print(f"\n❌ Estudiantes con errores:")
        for est in errores:
            print(f"   ✗ {est}")
    
    # Verificar base de datos después del autograding
    print("\n" + "="*70)
    print("🔍 VERIFICANDO BASE DE DATOS")
    print("="*70)
    # Cambiar al directorio del curso para el comando db
    os.chdir(BASE_PATH)
    !nbgrader db assignment list
    os.chdir('/content')
    print("="*70)

# Generar feedback para todos los estudiantes
import os

print(f"🚀 Generando feedback HTML para: {ASSIGNMENT_ID}\n")
print("="*70)

# IMPORTANTE: Cambiar al directorio del curso para que nbgrader encuentre el config
os.chdir(BASE_PATH)

# Verificar que existen archivos autograded (prerequisito)
autograded_count = 0
for estudiante in estudiantes:
    autograded_path = os.path.join(DIRS['autograded'], estudiante, ASSIGNMENT_ID, f"{ASSIGNMENT_ID}.ipynb")
    if os.path.exists(autograded_path):
        autograded_count += 1

print(f"📊 Archivos autograded encontrados: {autograded_count}/{len(estudiantes)}")

if autograded_count == 0:
    print("\n❌ ERROR: No hay archivos autograded")
    print("💡 Debes ejecutar primero la celda 25 (Autograding Masivo)")
    os.chdir('/content')
else:
    print(f"\n✅ Generando feedback para {autograded_count} estudiantes...\n")
    
    # Generar feedback - usar --assignment para especificar el assignment
    !nbgrader feedback '{ASSIGNMENT_ID}'
    
    # Volver a /content
    os.chdir('/content')
    
    print("\n" + "="*70)
    print("🔍 VERIFICANDO FEEDBACK GENERADO")
    print("="*70)
    
    # Verificar que se crearon archivos de feedback
    feedback_base = DIRS['feedback']
    feedback_files = []
    
    if os.path.exists(feedback_base):
        for estudiante in estudiantes:
            feedback_dir = os.path.join(feedback_base, estudiante, ASSIGNMENT_ID)
            if os.path.exists(feedback_dir):
                html_files = [f for f in os.listdir(feedback_dir) if f.endswith('.html')]
                if html_files:
                    feedback_files.append((estudiante, len(html_files)))
                    print(f"✅ {estudiante:40} → {len(html_files)} archivo(s) HTML")
                else:
                    print(f"⚠️  {estudiante:40} → carpeta existe pero sin HTML")
            else:
                print(f"❌ {estudiante:40} → carpeta no existe")
    else:
        print(f"❌ El directorio feedback/ no existe: {feedback_base}")
    
    print("\n" + "="*70)
    print("📊 RESUMEN")
    print("="*70)
    print(f"Total estudiantes:           {len(estudiantes)}")
    print(f"Con archivos autograded:     {autograded_count}")
    print(f"Con feedback HTML generado:  {len(feedback_files)}")
    
    if len(feedback_files) > 0:
        print(f"\n✅ Feedback generado exitosamente en: {feedback_base}")
        print(f"\n💡 Los estudiantes pueden ver su feedback en:")
        print(f"   {feedback_base}/{{nombre_estudiante}}/{ASSIGNMENT_ID}/{{ASSIGNMENT_ID}}.html")
    else:
        print(f"\n❌ No se generaron archivos de feedback")
        print(f"\n🔧 Posibles causas:")
        print(f"   1. No se ejecutó el autograding (celda 25)")
        print(f"   2. El comando feedback falló")
        print(f"   3. Problema con los metadatos del notebook")
        print(f"\n💡 Revisa el output del comando arriba para ver errores")
    
    print("="*70)

In [ ]:
# Exportar calificaciones a CSV
import os

OUTPUT_CSV = f"notas_{COURSE_ID}_{ASSIGNMENT_ID}.csv"

print("="*70)
print("📊 EXPORTANDO CALIFICACIONES A CSV")
print("="*70)

# IMPORTANTE: Cambiar al directorio del curso para que nbgrader encuentre el config
os.chdir(BASE_PATH)

# Primero verificar la base de datos antes de exportar
print("\n🔍 Verificando base de datos antes de exportar...\n")
!nbgrader db assignment list

print("\n" + "="*70)
print(f"📥 Exportando a: {OUTPUT_CSV}")
print("="*70 + "\n")

# Exportar calificaciones (SIN --force porque export no acepta ese flag)
!nbgrader export --to '{OUTPUT_CSV}'

# Volver a /content
os.chdir('/content')

# Verificar que se creó el archivo (estará en BASE_PATH)
csv_path = os.path.join(BASE_PATH, OUTPUT_CSV)
if os.path.exists(csv_path):
    print("\n" + "="*70)
    print("✅ EXPORTACIÓN EXITOSA")
    print("="*70)
    print(f"Archivo: {csv_path}")
    
    # Leer y mostrar un resumen (sin preview completo)
    try:
        import pandas as pd
        df = pd.read_csv(csv_path)
        
        print(f"\n📊 Resumen:")
        print(f"   Total de registros: {len(df)}")
        print(f"   Assignments únicos: {df['assignment'].nunique() if 'assignment' in df.columns else 'N/A'}")
        print(f"   Estudiantes únicos: {df['student_id'].nunique() if 'student_id' in df.columns else 'N/A'}")
        
        # Mostrar estadísticas de puntajes
        if 'score' in df.columns and 'max_score' in df.columns:
            print(f"\n   Estadísticas de puntajes:")
            print(f"   - Puntaje promedio: {df['score'].mean():.2f}/{df['max_score'].mean():.2f}")
            print(f"   - Puntaje máximo: {df['score'].max():.2f}")
            print(f"   - Puntaje mínimo: {df['score'].min():.2f}")
            
            # Advertir si hay muchos ceros (posible problema)
            zeros_count = (df['score'] == 0).sum()
            if zeros_count > 0:
                print(f"\n   ⚠️  Advertencia: {zeros_count} estudiantes con puntaje 0")
                print(f"      Verifica que el autograding se ejecutó correctamente")
                print(f"      Si es incorrecto, usa la celda 11.1 de diagnóstico")
    except Exception as e:
        print(f"\n   ⚠️  No se pudo leer el CSV para mostrar resumen: {e}")
    
    # Copiar a /content para facilitar descarga
    import shutil
    output_csv_content = f"/content/{OUTPUT_CSV}"
    shutil.copy(csv_path, output_csv_content)
    
    # Descargar archivo
    print(f"\n📥 Descargando archivo...")
    from google.colab import files
    files.download(output_csv_content)
    print(f"✅ Descarga completada")
    print(f"\n💡 Revisa el archivo descargado para ver las calificaciones completas")
    print("="*70)
else:
    print("\n" + "="*70)
    print("❌ ERROR EN LA EXPORTACIÓN")
    print("="*70)
    print("No se pudo crear el archivo de exportación")
    print(f"\nSe buscó en: {csv_path}")
    print("\n💡 Posibles causas:")
    print("   1. No se ejecutó el autograding correctamente")
    print("   2. La base de datos está vacía")
    print("   3. No hay submissions en el directorio autograded/")
    print("\n🔧 Soluciones:")
    print("   1. Ejecuta la celda de Autograding (celda 25) nuevamente")
    print("   2. Verifica que existan archivos en autograded/")
    print("   3. Usa la celda 11.1 de diagnóstico para más detalles")
    print("="*70)

## 11. Exportar Notas a CSV

In [ ]:
# DIAGNÓSTICO DE PROBLEMAS DE PUNTAJES
import subprocess
import os

print("="*70)
print("🔍 DIAGNÓSTICO DE PROBLEMAS CON PUNTAJES")
print("="*70)

# IMPORTANTE: Cambiar al directorio del curso para que nbgrader encuentre el config
os.chdir(BASE_PATH)

# 1. Verificar base de datos actual
print("\n1️⃣ Verificando base de datos de nbgrader...\n")
!nbgrader db assignment list

# Volver a /content
os.chdir('/content')

# 2. Verificar que existen archivos autograded
print("\n" + "="*70)
print("2️⃣ Verificando archivos en autograded/\n")
autograded_dir = DIRS['autograded']
if os.path.exists(autograded_dir):
    autograded_count = 0
    for root, dirs, files in os.walk(autograded_dir):
        nb_files = [f for f in files if f.endswith('.ipynb')]
        autograded_count += len(nb_files)
    print(f"   Notebooks autograded encontrados: {autograded_count}")
    if autograded_count == 0:
        print(f"   ⚠️  No hay archivos autograded - ejecuta la celda 22 primero")
else:
    print(f"   ❌ El directorio autograded/ no existe")

# 3. Opción para re-ejecutar autograding con debug
print("\n" + "="*70)
print("3️⃣ Opciones de corrección\n")
print("¿Qué deseas hacer?")
print("  1. Re-ejecutar autograding para UN estudiante (con debug)")
print("  2. Re-ejecutar autograding para TODOS (con --force)")
print("  3. Ver detalles de un estudiante específico")
print("  4. Limpiar base de datos y empezar de nuevo")
print("  0. No hacer nada")

opcion = input("\nSelecciona una opción: ").strip()

if opcion == '1':
    print("\n" + "="*70)
    print("Re-ejecutar autograding para UN estudiante")
    print("="*70)
    print(f"\nEstudiantes disponibles:")
    for i, est in enumerate(estudiantes, 1):
        print(f"  {i}. {est}")
    
    est_num = input(f"\nNúmero del estudiante (1-{len(estudiantes)}): ").strip()
    try:
        idx = int(est_num) - 1
        if 0 <= idx < len(estudiantes):
            estudiante = estudiantes[idx]
            print(f"\n🚀 Re-autograding con debug: {estudiante}\n")
            print("="*70)
            # Cambiar al directorio del curso
            os.chdir(BASE_PATH)
            !nbgrader autograde '{ASSIGNMENT_ID}' --student '{estudiante}' --force --debug
            print("="*70)
            
            # Verificar resultado
            print(f"\n✅ Verificando base de datos...\n")
            !nbgrader db assignment list --student '{estudiante}'
            # Volver a /content
            os.chdir('/content')
        else:
            print(f"❌ Número inválido")
    except:
        print(f"❌ Entrada inválida")

elif opcion == '2':
    print("\n" + "="*70)
    print("⚠️  ADVERTENCIA: Esto re-ejecutará el autograding para TODOS los estudiantes")
    print("="*70)
    confirmar = input("\n¿Estás seguro? (escribe 'SI' para confirmar): ").strip()
    
    if confirmar == 'SI':
        print(f"\n🚀 Re-autograding masivo con --force...\n")
        errores = []
        exitosos = []
        
        # Cambiar al directorio del curso
        os.chdir(BASE_PATH)
        
        for i, estudiante in enumerate(submissions_encontradas, 1):
            print(f"[{i}/{len(submissions_encontradas)}] {estudiante}... ", end='')
            try:
                cmd = f"nbgrader autograde '{ASSIGNMENT_ID}' --student '{estudiante}' --force"
                result = subprocess.run(cmd, shell=True, capture_output=True, text=True)
                if result.returncode == 0:
                    exitosos.append(estudiante)
                    print("✅")
                else:
                    errores.append(estudiante)
                    print(f"❌ (código {result.returncode})")
            except Exception as e:
                errores.append(estudiante)
                print(f"❌ ({e})")
        
        # Volver a /content
        os.chdir('/content')
        
        print("\n" + "="*70)
        print(f"✅ Exitosos: {len(exitosos)}/{len(submissions_encontradas)}")
        print(f"❌ Errores:  {len(errores)}/{len(submissions_encontradas)}")
        print("="*70)
        
        if exitosos:
            print(f"\n✅ Ahora puedes exportar nuevamente las notas (celda 26)")
    else:
        print("Cancelado")

elif opcion == '3':
    print("\n" + "="*70)
    print("Ver detalles de estudiante")
    print("="*70)
    estudiante = input(f"\nNombre del estudiante: ").strip()
    
    if estudiante in estudiantes:
        print(f"\n📊 Detalles de: {estudiante}\n")
        print("="*70)
        
        # Ver info en base de datos
        print("Base de datos:")
        os.chdir(BASE_PATH)
        !nbgrader db assignment list --student '{estudiante}'
        os.chdir('/content')
        
        # Ver archivos
        submitted_path = os.path.join(DIRS['submitted'], estudiante, ASSIGNMENT_ID)
        autograded_path = os.path.join(DIRS['autograded'], estudiante, ASSIGNMENT_ID)
        
        print(f"\n📁 Archivos:")
        print(f"  Submitted:  {os.path.exists(submitted_path)} ({submitted_path})")
        print(f"  Autograded: {os.path.exists(autograded_path)} ({autograded_path})")
        
        if os.path.exists(autograded_path):
            nb_path = os.path.join(autograded_path, f"{ASSIGNMENT_ID}.ipynb")
            if os.path.exists(nb_path):
                print(f"\n  ✓ Notebook autograded existe: {nb_path}")
            else:
                print(f"\n  ✗ Notebook autograded NO existe")
        print("="*70)
    else:
        print(f"❌ '{estudiante}' no está en la lista de estudiantes")

elif opcion == '4':
    print("\n" + "="*70)
    print("⚠️  ADVERTENCIA: Esto eliminará toda la base de datos de nbgrader")
    print("Tendrás que re-ejecutar el autograding desde cero")
    print("="*70)
    confirmar = input("\n¿Estás COMPLETAMENTE seguro? (escribe 'BORRAR TODO'): ").strip()
    
    if confirmar == 'BORRAR TODO':
        db_path = os.path.join(BASE_PATH, 'gradebook.db')
        if os.path.exists(db_path):
            os.remove(db_path)
            print(f"\n✅ Base de datos eliminada: {db_path}")
            print(f"\n💡 Ahora ejecuta la celda 22 para re-autograding")
        else:
            print(f"\n⚠️  No se encontró base de datos en: {db_path}")
    else:
        print("Cancelado")

else:
    print("\nCancelado")

print("\n" + "="*70)

## 12. Utilidades de Mantenimiento

### 12.1 Eliminar Submission de Estudiante Específico

In [ ]:
# Eliminar submission específica (para permitir re-submission)
print("⚠️  Esta acción eliminará la submission de un estudiante de la base de datos")
print("   (Permitirá que el estudiante vuelva a entregar)\n")

estudiante_eliminar = input(f"Nombre del estudiante (o ENTER para cancelar): ").strip()

if estudiante_eliminar:
    if estudiante_eliminar in estudiantes:
        !nbgrader db student remove '{estudiante_eliminar}' --assignment '{ASSIGNMENT_ID}' --force
        print(f"\n✅ Submission eliminada para: {estudiante_eliminar}")
    else:
        print(f"\n❌ Error: '{estudiante_eliminar}' no está en la lista de estudiantes")
else:
    print("Cancelado")

### 12.2 Limpiar Directorios

In [ ]:
# Función para limpiar un directorio
def limpiar_directorio(dir_path, dir_name):
    """Elimina todo el contenido de un directorio pero no el directorio mismo"""
    if not os.path.exists(dir_path):
        print(f"⚠️  El directorio {dir_name} no existe")
        return
    
    items_eliminados = 0
    for item in os.listdir(dir_path):
        item_path = os.path.join(dir_path, item)
        try:
            if os.path.isfile(item_path) or os.path.islink(item_path):
                os.unlink(item_path)
            elif os.path.isdir(item_path):
                shutil.rmtree(item_path)
            items_eliminados += 1
        except Exception as e:
            print(f"   ✗ Error eliminando {item}: {e}")
    
    print(f"✅ {dir_name}: {items_eliminados} items eliminados")

# Limpiar directorios (CUIDADO: Esto elimina archivos)
print("⚠️  ADVERTENCIA: Esto eliminará el contenido de los directorios\n")
print("Opciones:")
print("  1. Limpiar feedback")
print("  2. Limpiar autograded")
print("  3. Limpiar release")
print("  4. Limpiar TODO (feedback + autograded + release)")
print("  0. Cancelar\n")

opcion = input("Selecciona una opción: ").strip()

if opcion == '1':
    limpiar_directorio(DIRS['feedback'], 'feedback')
elif opcion == '2':
    limpiar_directorio(DIRS['autograded'], 'autograded')
elif opcion == '3':
    limpiar_directorio(DIRS['release'], 'release')
elif opcion == '4':
    print("\nLimpiando todos los directorios...\n")
    limpiar_directorio(DIRS['feedback'], 'feedback')
    limpiar_directorio(DIRS['autograded'], 'autograded')
    limpiar_directorio(DIRS['release'], 'release')
    print("\n✅ Limpieza completada")
else:
    print("Cancelado")

### 12.3 Ver Estadísticas del Assignment

In [ ]:
# Verificar estado de submissions
submitted_base = DIRS['submitted']

print(f"📊 Estado de submissions para: {ASSIGNMENT_ID}")
print("="*70)

submissions_ok = []
submissions_faltantes = []

for estudiante in estudiantes:
    assignment_path = os.path.join(submitted_base, estudiante, ASSIGNMENT_ID)
    
    if os.path.exists(assignment_path):
        files_in_dir = os.listdir(assignment_path)
        nb_files = [f for f in files_in_dir if f.endswith('.ipynb')]
        
        if nb_files:
            submissions_ok.append(estudiante)
            print(f"✓ {estudiante:40} → {len(nb_files)} archivo(s)")
        else:
            submissions_faltantes.append(estudiante)
            print(f"⚠  {estudiante:40} → carpeta vacía")
    else:
        submissions_faltantes.append(estudiante)
        print(f"✗ {estudiante:40} → no existe carpeta")

print("\n" + "="*70)
print("📊 RESUMEN")
print("="*70)
print(f"Total estudiantes:    {len(estudiantes)}")
print(f"✓ Con submissions:    {len(submissions_ok)} ({len(submissions_ok)/len(estudiantes)*100:.1f}%)")
print(f"✗ Sin submissions:    {len(submissions_faltantes)} ({len(submissions_faltantes)/len(estudiantes)*100:.1f}%)")

if submissions_faltantes:
    print(f"\n⚠️  Estudiantes sin submission:")
    for est in submissions_faltantes:
        print(f"   - {est}")

print("="*70)

## 13. Resumen de Configuración

Vista rápida de todos los parámetros configurados en esta sesión.

In [ ]:
# Resumen de configuración
print("="*70)
print("📋 RESUMEN DE CONFIGURACIÓN")
print("="*70)
print(f"\n🎓 Curso:")
print(f"   ID:              {COURSE_ID}")
print(f"   Assignment:      {ASSIGNMENT_ID}")
print(f"   Ruta base:       {BASE_PATH}")
print(f"   Timeout:         {TIMEOUT}s")

print(f"\n👥 Estudiantes:")
print(f"   Total:           {len(estudiantes)}")
print(f"   Con submissions: {len(submissions_ok) if 'submissions_ok' in locals() else 'N/A'}")

print(f"\n📁 Directorios:")
for name, path in DIRS.items():
    exists = "✓" if os.path.exists(path) else "✗"
    print(f"   {exists} {name:12} → {path}")

print(f"\n📄 Archivos:")
print(f"   Config:          /content/nbgrader_config.py")
print(f"   CSV estudiantes: {csv_filename if 'csv_filename' in locals() else 'N/A'}")
print(f"   Notas export:    {OUTPUT_CSV if 'OUTPUT_CSV' in locals() else 'N/A'}")

print("\n" + "="*70)
print("✅ Sistema configurado y listo")
print("="*70)